# Step 16 -- CARE-Sim Control Layer

Runs the CARE-Sim decision layer on Google Colab:
- planner baseline over the 16 drug combinations
- simulator-based DDQN training
- held-out evaluation against planner, DDQN, random, and repeat-last baselines

## Before running

**Step 1:** From the repo root, rebuild the upload zip locally:
```\npython scripts/prepare_colab_upload.py\n```

**Step 2:** Upload to Google Drive:
- `caresim_colab.zip` -> `MyDrive/CareAI/`
- `data/processed/icu_readmit/rl_dataset_tier2.parquet` -> `MyDrive/CareAI/data/`
- trained CARE-Sim folder `models/icu_readmit/caresim/` -> `MyDrive/CareAI/models/icu_readmit/caresim/`

**Step 3:** Runtime -> Change runtime type -> **T4 GPU**.


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Step 16 will be much slower on CPU.')


In [ ]:
# Cell 2: Unzip source files + verify inputs
import os, sys, zipfile, shutil

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_PATH   = os.path.join(DRIVE_ROOT, 'caresim_colab.zip')
UNZIP_DIR  = '/content/caresim_src'
SRC_PATH   = os.path.join(UNZIP_DIR, 'src')

DATA_PATH         = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_tier2.parquet')
CARESIM_MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_causal')
CONTROL_MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_control_causal')
REPORT_DIR        = os.path.join(DRIVE_ROOT, 'reports', 'icu_readmit', 'caresim_control_causal')

CONTROL_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_16_caresim_control_causal.py')

for path, name in [(ZIP_PATH, 'caresim_colab.zip'), (DATA_PATH, 'rl_dataset_tier2.parquet')]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{name} not found at {path}')

if not os.path.exists(CARESIM_MODEL_DIR):
    raise FileNotFoundError(f'Trained CARE-Sim ensemble not found at {CARESIM_MODEL_DIR}')

if os.path.exists(UNZIP_DIR):
    shutil.rmtree(UNZIP_DIR)

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)
print(f'Unzipped to {UNZIP_DIR}')

for path in [SRC_PATH, CONTROL_SCRIPT]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing expected extracted file: {path}')

os.makedirs(CONTROL_MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print('Inputs ready:')
print(f'  Data:    {DATA_PATH}')
print(f'  CARE-Sim:{CARESIM_MODEL_DIR}')
print(f'  Control: {CONTROL_MODEL_DIR}')
print(f'  Reports: {REPORT_DIR}')


In [ ]:
# Cell 3: Planner-only evaluation (quick baseline)
import subprocess

def run_streaming(cmd, env):
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

PLANNER_ARGS = [
    sys.executable, '-u', CONTROL_SCRIPT, 'planner',
    '--data', DATA_PATH,
    '--model-dir', CARESIM_MODEL_DIR,
    '--report-dir', REPORT_DIR,
    '--history-len', '5',
    '--observation-window', '5',
    '--rollout-steps', '5',
    '--planner-horizon', '3',
    '--episodes-per-split', '50',
    '--uncertainty-penalty', '0.25',
    '--device', device,
]

ENV = {**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'}
print('=' * 60)
print('STEP 16 -- Planner Baseline')
print('=' * 60)
rc = run_streaming(PLANNER_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Planner baseline FAILED.')
print('Planner baseline complete.')


In [ ]:
# Cell 4: DDQN training against CARE-Sim
# Updated defaults: larger training budget + slower epsilon decay to reduce action collapse.

TRAIN_EPISODES = 2000
TRAIN_STEPS = 20000
BATCH_SIZE = 64
TARGET_SYNC_EVERY = 200
EPSILON_END = 0.10
EPSILON_DECAY_STEPS = 20000

DDQN_ARGS = [
    sys.executable, '-u', CONTROL_SCRIPT, 'train-ddqn',
    '--data', DATA_PATH,
    '--model-dir', CARESIM_MODEL_DIR,
    '--control-model-dir', CONTROL_MODEL_DIR,
    '--history-len', '5',
    '--observation-window', '5',
    '--rollout-steps', '5',
    '--planner-horizon', '3',
    '--train-episodes', str(TRAIN_EPISODES),
    '--train-steps', str(TRAIN_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--target-sync-every', str(TARGET_SYNC_EVERY),
    '--epsilon-end', str(EPSILON_END),
    '--epsilon-decay-steps', str(EPSILON_DECAY_STEPS),
    '--warmup-steps', '256',
    '--replay-capacity', '20000',
    '--uncertainty-penalty', '0.25',
    '--device', device,
]

print('=' * 60)
print('STEP 16 -- DDQN Training')
print('=' * 60)
rc = run_streaming(DDQN_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('DDQN training FAILED.')
print('DDQN training complete.')


In [ ]:
# Cell 5: Evaluate planner vs DDQN vs baselines
DDQN_PATH = os.path.join(CONTROL_MODEL_DIR, 'ddqn_model.pt')
if not os.path.exists(DDQN_PATH):
    raise FileNotFoundError(f'DDQN checkpoint not found at {DDQN_PATH}')

EVAL_ARGS = [
    sys.executable, '-u', CONTROL_SCRIPT, 'eval',
    '--data', DATA_PATH,
    '--model-dir', CARESIM_MODEL_DIR,
    '--report-dir', REPORT_DIR,
    '--ddqn-path', DDQN_PATH,
    '--history-len', '5',
    '--observation-window', '5',
    '--rollout-steps', '5',
    '--planner-horizon', '3',
    '--episodes-per-split', '100',
    '--uncertainty-penalty', '0.25',
    '--device', device,
]

print('=' * 60)
print('STEP 16 -- Policy Evaluation')
print('=' * 60)
rc = run_streaming(EVAL_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Step 16 evaluation FAILED.')

summary_path = os.path.join(REPORT_DIR, 'step_16_summary.json')
print('\nSaved summary:')
print(summary_path)
print(open(summary_path, 'r', encoding='utf-8').read())


In [ ]:
# Cell 6: Verify outputs on Drive
print('Control model files saved to Drive:')
for root, dirs, files in os.walk(CONTROL_MODEL_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, CONTROL_MODEL_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nReport files saved to Drive:')
for root, dirs, files in os.walk(REPORT_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, REPORT_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nDownload from Drive to these local repo folders:')
print('  CareAI/models/icu_readmit/caresim_control/')
print('  CareAI/reports/icu_readmit/caresim_control/')
